## CABRA Evaluations: Analysis Notebook

This Jupyter Notebook contains the main evaluations of our paper. In particular, we provide scripts to recreate the following plots:
- Evaluating LLM and Coding Agent accuracy on CABRA 
- Evaluating LLM and Coding Agent accuracy on the Merge Codebases task
- Tool call analysis on CABRA 
- Tool call analysis on the Merge Codebases task

### Dataset Collection

First, collect all of the model/agent runs

In [ ]:
from __future__ import annotations

import gzip
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import tqdm

EXPERIMENT_DATA_ROOT = Path("../local_data")
RESULTS_ROOT = EXPERIMENT_DATA_ROOT / "results"
RUN_KEYS = ["run_name", "task_set_name", "task_type", "model", "dag_id"]

TASK_TYPES = [
    "dead_code",
    "add_parameter",
    "add_return_value",
    "cache_function",
    "extract_helper",
]

EXPERIMENTS = {
    "function_traversal": {
        "name": "Function Traversal",
        "n_pattern": re.compile(r"_en=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "score_kinds": ("behavioral",),
        "dead_code_kinds": ("behavioral", "function_existence"),
        "order": 0,
    },
    "function_search": {
        "name": "Function Search",
        "n_pattern": re.compile(r"_cn=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "score_kinds": ("behavioral",),
        "dead_code_kinds": ("behavioral", "function_existence"),
        "order": 1,
    },
    "add_instructions": {
        "name": "Instruction Following",
        "n_pattern": re.compile(r"_nc=\((?P<lo>[^,]+),(?P<hi>[^)]+)\)"),
        "score_kinds": ("behavioral", "constraint_structure"),
        "dead_code_kinds": ("behavioral", "function_existence"),
        "order": 2,
    },
    "runtime_resolution": {
        "name": "Runtime Resolution",
        "n_pattern": re.compile(r"_ifhops=(?P<value>[^_]+)"),
        "score_kinds": ("behavioral", "branch_placement"),
        "dead_code_kinds": ("behavioral", "branch_placement", "function_existence"),
        "order": 3,
    },
    "merge_codebases": {
        "name": "Merge Codebases",
        "n_pattern": re.compile(r"_n=(?P<value>[^_]+)"),
        "score_kinds": ("behavioral", "difference_blocks"),
        "dead_code_kinds": ("behavioral", "difference_blocks"),
        "order": 4,
    },
}


def open_jsonl(path: Path):
    return gzip.open(path, "rt", encoding="utf-8") if path.suffix == ".gz" else path.open(encoding="utf-8")


def experiment_key_from_run_name(run_name: str) -> str | None:
    for key in EXPERIMENTS:
        if run_name.startswith(f"{key}_"):
            return key
    return None


def n_value_from_run_name(run_name: str, experiment_key: str) -> float | None:
    pattern = EXPERIMENTS[experiment_key]["n_pattern"]
    match = pattern.search(run_name)
    if match is None:
        return None
    if "value" in match.groupdict():
        return float(match.group("value"))
    lo = float(match.group("lo"))
    hi = float(match.group("hi"))
    return (lo + hi) / 2


def kind_passes(summary: dict, kind: str) -> bool:
    counts = summary.get(kind)
    if counts is None:
        return False
    return (
        counts.get("pass", 0) > 0
        and counts.get("fail", 0) == 0
        and counts.get("error", 0) == 0
    )


def record_passes(record: dict, experiment_key: str, task_type: str) -> bool:
    if record.get("skipped"):
        return False
    spec = EXPERIMENTS[experiment_key]
    score_kinds = spec["dead_code_kinds"] if task_type == "dead_code" else spec["score_kinds"]
    summary = record.get("summary") or {}
    return all(kind_passes(summary, kind) for kind in score_kinds)


def model_group(model: str) -> str:
    return model.split("/", 1)[0]


def collect_model_scores(results_root: Path = RESULTS_ROOT) -> pd.DataFrame:
    if not results_root.exists():
        raise FileNotFoundError(f"results directory not found: {results_root}")

    rows = []
    paths = list(results_root.glob("**/*.jsonl")) + list(results_root.glob("**/*.jsonl.gz"))
    for path in tqdm.tqdm(sorted(paths), desc="Loading scores", unit="file"):
        rel_parts = path.relative_to(results_root).parts
        if len(rel_parts) < 3:
            continue
        run_name = rel_parts[0]
        if rel_parts[1] in {"code", "copilot"}:
            task_set_name = run_name
            task_type = "merge_codebases"
            model_parts = rel_parts[1:]
        elif len(rel_parts) >= 5:
            task_set_name = rel_parts[1]
            task_type = rel_parts[2]
            model_parts = rel_parts[3:]
        else:
            continue
        if task_type not in TASK_TYPES:
            if task_type != "merge_codebases":
                continue
        experiment_key = experiment_key_from_run_name(run_name)
        if experiment_key is None:
            continue
        n_value = n_value_from_run_name(run_name, experiment_key)
        if n_value is None:
            continue

        model = "/".join(model_parts).removesuffix(".gz").removesuffix(".jsonl")
        group = model_group(model)
        spec = EXPERIMENTS[experiment_key]

        with open_jsonl(path) as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                rows.append(
                    {
                        "task_name": spec["name"],
                        "N": n_value,
                        "model": model,
                        "score": int(record_passes(record, experiment_key, task_type)),
                        "task_set_name": task_set_name,
                        "task_type": task_type,
                        "dag_id": str(record.get("dag_id") or ""),
                        "model_group": group,
                        "run_name": run_name,
                        "experiment_order": spec["order"],
                        "source_path": str(path),
                    }
                )

    if not rows:
        raise ValueError(f"no score rows found under {results_root}")

    return pd.DataFrame(rows).sort_values(
        ["experiment_order", "N", "model_group", "model", "task_set_name", "task_type"]
    )

In [ ]:
records_df = collect_model_scores()
df = records_df[["task_name", "N", "score", *RUN_KEYS]].copy()

df.head()

### Load the precomputed minimal tool-call summary

The `tool_call_token_counts.jsonl` file contains only `(run_name, task_set_name, task_type, model, dag_id)`, `tool_call_index`, `label`, and `tool_call_tokens`. Generate it separately with `summarize_tool_call_tokens.py`; all tool-call analysis below uses only this summary file, so the original traces and their contents are not needed by the notebook.

In [ ]:
TOOL_CALL_TOKEN_COUNTS_PATH = EXPERIMENT_DATA_ROOT / "tool_call_results" / "tool_call_token_counts.jsonl"


def collect_tool_calls(summary_path: Path = TOOL_CALL_TOKEN_COUNTS_PATH) -> pd.DataFrame:
    if not summary_path.exists():
        raise FileNotFoundError(f"tool-call token summary not found: {summary_path}")

    rows = []
    with open_jsonl(summary_path) as file:
        for line in tqdm.tqdm(file, desc="Loading tool-call summary", unit="call"):
            line = line.strip()
            if line:
                record = json.loads(line)
                record["run_name"] = (record.get("run_name") or "").replace(
                    "add_instructions_fixed_", "add_instructions_", 1
                )
                rows.append(record)

    if not rows:
        raise ValueError(f"no tool-call rows found in {summary_path}")
    calls_df = pd.DataFrame(rows).sort_values([*RUN_KEYS, "tool_call_index"])
    required_columns = {*RUN_KEYS, "tool_call_index", "label", "tool_call_tokens"}
    missing_columns = required_columns - set(calls_df.columns)
    if missing_columns:
        raise ValueError(f"tool-call summary lacks columns: {sorted(missing_columns)}")
    if calls_df.duplicated([*RUN_KEYS, "tool_call_index"]).any():
        raise ValueError("duplicate tool-call indices for the same run identifiers")

    return calls_df.groupby(RUN_KEYS, as_index=False).agg(
        tool_calls=("label", list),
        tool_call_tokens=("tool_call_tokens", list),
    )


tool_call_df = collect_tool_calls()

tool_call_df.head()

### LLM vs Agent Accuracy Plots

In [ ]:
# Shared plotting logic

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TASK_X_LABELS = {
    "Function Traversal": "Number of Edit Functions (N)",
    "Function Search": "Number of Non-Edit Functions (N)",
    "Instruction Following": "Number of Instructions (N)",
    "Runtime Resolution": "Number of Runtime Steps (N)",
    "Merge Codebases": "Number of Functions (N)",
}
TASK_NAME_ORDER = {spec["name"]: spec["order"] for spec in EXPERIMENTS.values()}
MODEL_GROUP_ORDER = ["code", "copilot"]
ROW_LABELS = {
    "code": "LLM Accuracy",
    "copilot": "Agent Accuracy",
}
PRIORITY_MODEL_COLORS = {
    "GPT-5.5": "tab:blue",
    "GPT-5.4 Mini": "tab:red",
}
PRIORITY_LEGEND_LABELS = ["GPT-5.4 Mini", "GPT-5.5"]


def model_label(model: str) -> str:
    name = model.split("/", 1)[-1]
    name = name.removeprefix("openai/")
    name = re.sub(r"_20\d\d-\d\d-\d\d$", "", name)
    replacements = {
        "gpt-5.3-codex": "GPT-5.3 Codex",
        "gpt-5": "GPT-5",
        "grok-4-1-fast-reasoning_1": "Grok 4.1 Fast",
        "DeepSeek-V3.2_1": "DeepSeek V3.2",
        "gpt-oss-120b_1": "GPT OSS 120B",
        "Mistral-Large-3_1": "Mistral Large 3",
        "claude-sonnet-4.6": "Sonnet 4.6",
        "claude-opus-4.7": "Opus 4.7",
        "gpt-5.5": "GPT-5.5",
        "gpt-5.4-mini": "GPT-5.4 Mini",
        "gemini-3.1-pro-preview": "Gemini 3.1 Pro",
        "gemini-3.5-flash": "Gemini 3.5 Flash",
    }
    return replacements.get(name, name.replace("_", " ").replace("-", " ").title())


def format_axis_value(value: float) -> str:
    return str(int(value)) if float(value).is_integer() else f"{value:g}"


def legend_model_order(models: list[str]) -> list[str]:
    priority = {label: idx for idx, label in enumerate(PRIORITY_LEGEND_LABELS)}
    return sorted(models, key=lambda model: (priority.get(model_label(model), len(priority)), models.index(model)))


def aggregate_scores_for_scale_plot(data: pd.DataFrame, tasks_to_plot: list) -> pd.DataFrame:
    task_order = {task_name: index for index, task_name in enumerate(tasks_to_plot)}
    plot_records = data.copy()
    plot_records = plot_records[plot_records["task_name"].isin(tasks_to_plot)]
    plot_records["model_group"] = plot_records["model"].map(model_group)
    plot_records["experiment_order"] = plot_records["task_name"].map(task_order)
    plot_records["x_label"] = plot_records["task_name"].map(TASK_X_LABELS)

    plot_data = (
        plot_records.groupby(
            ["task_name", "N", "model", "model_group", "experiment_order", "x_label"],
            as_index=False,
        )
        .agg(passes=("score", "sum"), total=("score", "size"))
        .sort_values(["experiment_order", "N", "model_group", "model"])
    )
    plot_data["score"] = plot_data["passes"] / plot_data["total"]
    plot_data["stderr"] = np.sqrt(plot_data["score"] * (1 - plot_data["score"]) / plot_data["total"])
    return plot_data


def global_model_colors(plot_data: pd.DataFrame) -> dict[str, dict[str, tuple]]:
    ordered_labels = []
    for model in plot_data["model"].drop_duplicates():
        label = model_label(model)
        if label not in ordered_labels:
            ordered_labels.append(label)

    priority_rgb = {mcolors.to_rgb(color) for color in PRIORITY_MODEL_COLORS.values()}
    palette = [color for color in plt.get_cmap("tab20").colors if color not in priority_rgb]
    label_colors = {
        label: palette[index % len(palette)]
        for index, label in enumerate(label for label in ordered_labels if label not in PRIORITY_MODEL_COLORS)
    }
    label_colors.update(
        {
            label: mcolors.to_rgba(color)
            for label, color in PRIORITY_MODEL_COLORS.items()
            if label in ordered_labels
        },
    )

    colors = {}
    for group in MODEL_GROUP_ORDER:
        models = plot_data.loc[plot_data["model_group"] == group, "model"].drop_duplicates()
        colors[group] = {model: label_colors[model_label(model)] for model in models}
    return colors


def make_scale_task_line_plot(plot_data: pd.DataFrame) -> None:
    task_panels = (
        plot_data[["task_name", "experiment_order", "x_label"]]
        .drop_duplicates()
        .sort_values("experiment_order")
        .to_dict("records")
    )
    colors = global_model_colors(plot_data)

    fig, axes = plt.subplots(
        len(MODEL_GROUP_ORDER),
        len(task_panels),
        figsize=(max(3.0 * len(task_panels), 6), 4.0),
        squeeze=False,
    )

    for col, task_panel in enumerate(task_panels):
        task_df = plot_data[plot_data["task_name"] == task_panel["task_name"]]
        x_values = sorted(task_df["N"].unique())

        for row, group in enumerate(MODEL_GROUP_ORDER):
            ax = axes[row][col]
            group_df = task_df[task_df["model_group"] == group]
            if group_df.empty:
                ax.axis("off")
                continue

            for model in group_df["model"].drop_duplicates():
                model_df = group_df[group_df["model"] == model].sort_values("N")
                ax.errorbar(
                    model_df["N"],
                    model_df["score"],
                    yerr=model_df["stderr"],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.8,
                    capsize=2.5,
                    color=colors[group][model],
                    label=model_label(model),
                )

            ax.set_xscale("log")
            ax.set_xlim(min(x_values) / 1.08, max(x_values) * 1.08)
            ax.set_ylim(-0.03, 1.03)
            ax.set_xticks(x_values)
            ax.set_xticklabels([format_axis_value(x) for x in x_values])
            ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
            ax.grid(True, alpha=0.25)

            if row == 0:
                ax.set_title(task_panel["task_name"], fontsize=13)
                ax.set_xlabel("")
                ax.tick_params(labelbottom=False)
            else:
                ax.set_xlabel(task_panel["x_label"], fontsize=9)

            if col == 0:
                ax.set_ylabel(ROW_LABELS[group])

            if col == len(task_panels) - 1:
                models = legend_model_order(list(group_df["model"].drop_duplicates()))
                handles = [
                    Line2D([0], [0], color=colors[group][model], marker="o", markersize=5, linewidth=1.8)
                    for model in models
                ]
                legend = ax.legend(
                    handles,
                    [model_label(model) for model in models],
                    title="Agents:" if group == "copilot" else "LLMs:",
                    loc="center left",
                    bbox_to_anchor=(1.02, 0.5),
                    fontsize=8,
                    title_fontsize=9,
                    frameon=True,
                )
                legend._legend_box.align = "left"

    fig.tight_layout(h_pad=2.5)
    plt.show()

#### CABRA Task Accuracy

In [ ]:
scale_task_line_df = aggregate_scores_for_scale_plot(df, ['Function Traversal', 'Function Search', 'Runtime Resolution', 'Instruction Following'])
make_scale_task_line_plot(scale_task_line_df)

In [ ]:
scale_task_line_df

#### Merge Codebases Task Accuracy

In [ ]:
scale_task_line_df = aggregate_scores_for_scale_plot(df, ['Merge Codebases'])
scale_task_line_df["x_label"] = "Total Lines of Code (N)"
make_scale_task_line_plot(scale_task_line_df)

### Tool Call Analysis

In [ ]:
import warnings

from matplotlib.ticker import MaxNLocator

TOOL_GROUPS = [
    ("understand", ("understand",), "Analyze"),
    ("read", ("read",), "Read"),
    ("search", ("search",), "Search"),
    ("edit", ("edit",), "Edit"),
    ("test", ("test",), "Test"),
    ("other", ("summary", "plan", "file", "other"), "Other"),
]
TOOL_METRIC_LABELS = {
    "num_tool_calls": "Mean Tool Calls",
    "tool_command_tokens_k": "Mean Tokens (K)",
}


def stderr(values: pd.Series) -> float:
    values = values.dropna()
    if len(values) <= 1:
        return 0.0 if len(values) == 1 else np.nan
    return float(values.std(ddof=1) / np.sqrt(len(values)))


def build_tool_call_plot_df(scores_df: pd.DataFrame, tool_runs_df: pd.DataFrame) -> pd.DataFrame:
    for label, data in (("scores", scores_df), ("tool calls", tool_runs_df)):
        if not set(RUN_KEYS).issubset(data.columns):
            raise ValueError(f"{label} lack run identifiers; rerun collection and rebuild plot_data.pkl")
        if data[RUN_KEYS].isna().any().any() or data[RUN_KEYS].eq("").any().any():
            raise ValueError(f"{label} contain missing run identifiers")

    score_meta = (
        scores_df[scores_df["model"].map(model_group) == "copilot"]
        [[*RUN_KEYS, "task_name", "N"]]
        .assign(model_group="copilot")
    )
    matched_runs = score_meta.merge(
        tool_runs_df, on=RUN_KEYS, how="outer", validate="one_to_one", indicator=True,
    )
    unmatched_scores = int(matched_runs["_merge"].eq("left_only").sum())
    unmatched_tools = int(matched_runs["_merge"].eq("right_only").sum())
    if unmatched_scores or unmatched_tools:
        warnings.warn(
            f"Tool analysis excludes {unmatched_scores} score runs without tool data and "
            f"{unmatched_tools} tool runs without scores; missing data are not counted as zero calls.",
            stacklevel=2,
        )
    matched_runs = matched_runs.loc[matched_runs["_merge"] == "both"].drop(columns="_merge")
    if matched_runs.empty:
        overlap = {
            key: len(set(score_meta[key]) & set(tool_runs_df[key]))
            for key in RUN_KEYS
        }
        examples = {
            key: {
                "score": score_meta[key].iloc[0] if not score_meta.empty else None,
                "tool": tool_runs_df[key].iloc[0] if not tool_runs_df.empty else None,
            }
            for key in RUN_KEYS
        }
        raise ValueError(f"no tool-call runs matched; per-column overlap={overlap}; examples={examples}")
    run_meta = matched_runs[[*RUN_KEYS, "task_name", "N", "model_group"]]

    calls = matched_runs.explode(["tool_calls", "tool_call_tokens"], ignore_index=True)
    calls = calls.rename(columns={"tool_calls": "tool_label", "tool_call_tokens": "tool_tokens"})
    calls["tool_tokens_k"] = pd.to_numeric(calls["tool_tokens"], errors="coerce") / 1000.0

    rows = []
    for group_key, labels, display in TOOL_GROUPS:
        group_calls = calls[calls["tool_label"].isin(labels)]
        count_metric = (
            group_calls.groupby(RUN_KEYS, as_index=False)
            .size()
            .rename(columns={"size": "num_tool_calls"})
        )
        token_metric = (
            group_calls.groupby(RUN_KEYS, as_index=False)["tool_tokens_k"]
            .sum(min_count=1)
            .rename(columns={"tool_tokens_k": "tool_command_tokens_k"})
        )
        run_values = run_meta.merge(
            count_metric, on=RUN_KEYS, how="left", validate="one_to_one",
        ).merge(
            token_metric, on=RUN_KEYS, how="left", validate="one_to_one",
        )
        run_values["num_tool_calls"] = run_values["num_tool_calls"].fillna(0)
        run_values["tool_command_tokens_k"] = run_values["tool_command_tokens_k"].fillna(0)
        run_values["tool_key"] = group_key
        run_values["tool_display"] = display
        rows.append(run_values)

    return pd.concat(rows, ignore_index=True)


def summarize_tool_metric(tool_plot_df: pd.DataFrame, group_cols: list[str], metric: str) -> pd.DataFrame:
    return (
        tool_plot_df.groupby(group_cols, as_index=False)[metric]
        .agg(mean="mean", stderr=stderr)
        .sort_values(group_cols)
    )


def active_tool_groups(tool_plot_df: pd.DataFrame, metric: str) -> list[tuple[str, str]]:
    active = (
        tool_plot_df.groupby(["tool_key", "tool_display"], as_index=False)[metric]
        .sum(min_count=1)
    )
    active = active[active[metric].fillna(0) > 0]
    order = {key: index for index, (key, _labels, _display) in enumerate(TOOL_GROUPS)}
    return [
        (row.tool_key, row.tool_display)
        for row in active.sort_values("tool_key", key=lambda column: column.map(order)).itertuples()
    ]


def style_tool_axis(ax, x_values: list[float]) -> None:
    ax.set_xscale("log")
    ax.set_xlim(min(x_values) / 1.08, max(x_values) * 1.08)
    ax.set_xticks(x_values)
    ax.set_xticklabels([format_axis_value(x) for x in x_values])
    ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, min_n_ticks=2))
    ax.grid(True, alpha=0.25)


tool_call_plot_df = build_tool_call_plot_df(df, tool_call_df)

#### Comparing Tool Use Across Tasks 

In [ ]:
def make_tool_call_plot_by_tool_column(tool_plot_df: pd.DataFrame, task_names: list, tool_names: list) -> None:
    metrics = ["tool_command_tokens_k", "num_tool_calls"]
    tool_plot_df = tool_plot_df[tool_plot_df["task_name"].isin(task_names)]
    active_task_names = set(tool_plot_df["task_name"])
    task_panels = [task_name for task_name in task_names if task_name in active_task_names]
    if not task_panels:
        raise ValueError(f"no active tasks matched task_names={task_names!r}")

    groups = active_tool_groups(tool_plot_df, "num_tool_calls")
    groups_by_name = {
        name.lower(): (group_key, display)
        for group_key, display in groups
        for name in (group_key, display)
    }
    groups = [groups_by_name[tool_name.lower()] for tool_name in tool_names if tool_name.lower() in groups_by_name]
    if not groups:
        raise ValueError(f"no active tool-call groups matched tool_names={tool_names!r}")

    task_colors = {
        task_name: plt.get_cmap("tab10")(index % 10)
        for index, task_name in enumerate(task_panels)
    }

    fig, axes = plt.subplots(len(metrics), len(groups), figsize=(2.4 * len(groups), 3.5), squeeze=False)
    for row, metric in enumerate(metrics):
        summary = summarize_tool_metric(tool_plot_df, ["task_name", "N", "tool_key", "tool_display"], metric)
        for col, (group_key, display) in enumerate(groups):
            ax = axes[row][col]
            group_summary = summary[summary["tool_key"] == group_key]
            for task_name in task_panels:
                line = group_summary[group_summary["task_name"] == task_name].sort_values("N")
                if line.empty or line["mean"].isna().all():
                    continue
                ax.errorbar(
                    line["N"],
                    line["mean"],
                    yerr=line["stderr"],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.8,
                    capsize=2.5,
                    color=task_colors[task_name],
                    label=task_name,
                )
            x_values = sorted(tool_plot_df["N"].unique())
            if row == 0:
                ax.set_title(display, fontsize=12)
            if col == 0:
                ax.set_ylabel(TOOL_METRIC_LABELS[metric])
            if row == len(metrics) - 1:
                ax.set_xlabel("Task Complexity (N)", fontsize=9)
            else:
                ax.tick_params(labelbottom=False)
            style_tool_axis(ax, x_values)

    handles = [
        Line2D([0], [0], color=task_colors[task_name], marker="o", markersize=5, linewidth=1.8)
        for task_name in task_panels
    ]
    fig.legend(
        handles,
        task_panels,
        title="Task",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.11),
        ncol=min(len(task_panels), 5),
        fontsize=9,
        title_fontsize=10,
    )
    fig.tight_layout(rect=(0.0, 0.10, 1.0, 1.0))
    plt.show()

In [ ]:
make_tool_call_plot_by_tool_column(tool_call_plot_df,
                                   ['Function Traversal', 'Function Search', 'Runtime Resolution', 'Instruction Following'],
                                   ['Analyze', 'Read', 'Search', 'Edit', 'Test'])

#### Comparing Tool Use Within Tasks

In [ ]:
def make_tool_call_plot(tool_plot_df: pd.DataFrame, task_names: list) -> None:
    metrics = ["num_tool_calls", "tool_command_tokens_k"]
    tool_plot_df = tool_plot_df[tool_plot_df["task_name"].isin(task_names)]
    task_panels = (
        tool_plot_df[["task_name"]]
        .drop_duplicates()
        .assign(experiment_order=lambda data: data["task_name"].map(TASK_NAME_ORDER))
        .sort_values("experiment_order")
    )
    groups = active_tool_groups(tool_plot_df, "num_tool_calls")
    cmap = plt.get_cmap("tab10")
    group_colors = {key: cmap(index % 10) for index, (key, _display) in enumerate(groups)}

    fig, axes = plt.subplots(len(metrics), len(task_panels), figsize=(3.0 * len(task_panels), 3.5), squeeze=False)
    for row, metric in enumerate(metrics):
        summary = summarize_tool_metric(tool_plot_df, ["task_name", "N", "tool_key", "tool_display"], metric)
        for col, task_name in enumerate(task_panels["task_name"]):
            ax = axes[row][col]
            task_summary = summary[summary["task_name"] == task_name]
            x_values = sorted(tool_plot_df.loc[tool_plot_df["task_name"] == task_name, "N"].unique())
            for group_key, display in groups:
                line = task_summary[task_summary["tool_key"] == group_key].sort_values("N")
                if line.empty or line["mean"].isna().all():
                    continue
                ax.errorbar(
                    line["N"],
                    line["mean"],
                    yerr=line["stderr"],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.8,
                    capsize=2.5,
                    color=group_colors[group_key],
                    label=display,
                )
            if row == 0:
                ax.set_title(task_name, fontsize=12)
            if col == 0:
                ax.set_ylabel(TOOL_METRIC_LABELS[metric])
            if row == len(metrics) - 1:
                ax.set_xlabel(TASK_X_LABELS[task_name], fontsize=9)
            else:
                ax.tick_params(labelbottom=False)
            style_tool_axis(ax, x_values)

    handles = [
        Line2D([0], [0], color=group_colors[key], marker="o", markersize=5, linewidth=1.8)
        for key, _display in groups
    ]
    fig.legend(
        handles,
        [display for _key, display in groups],
        title="Tool Call Type",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.10),
        ncol=min(len(groups), 6),
        fontsize=9,
        title_fontsize=10,
    )
    fig.tight_layout(rect=(0.0, 0.08, 1.0, 1.0))
    plt.show()

In [ ]:
make_tool_call_plot(tool_call_plot_df, ['Function Traversal', 'Function Search', 'Runtime Resolution', 'Instruction Following', 'Merge Codebases'])